In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG16, ResNet50, InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Concatenate
from tensorflow.keras.utils import to_categorical

In [ ]:
x = np.random.rand(100, 224, 224, 3).astype(np.float32)
y = to_categorical(np.random.randint(0, 2, 100),2)
x_val = np.random.rand(20, 224, 224, 3).astype(np.float32)
y_val = to_categorical(np.random.randint(0, 2, 20),2)

In [ ]:
base_model1 = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model2 = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model3 = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for n in [base_model1, base_model2, base_model3]:
    for layer in n.layers:
        layer.trainable = False

In [ ]:
x1 = GlobalAveragePooling2D()(base_model1.output)
x2 = GlobalAveragePooling2D()(base_model2.output)
x3 = GlobalAveragePooling2D()(base_model3.output)

In [ ]:
merged = Concatenate()([x1, x2, x3])
output = Dense(2, activation='softmax')(merged)

In [ ]:
target_model = Model(inputs=[base_model1.input, base_model2.input, base_model3.input], outputs=output)

In [ ]:
target_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], run_eagerly=True)
target_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]

 Total params: 60,114,402 (229.32 MB)

 Trainable params: 9,218 (36.01 KB)

 Non-trainable params: 60,105,184 (229.28 MB)

In [ ]:
target_model.fit(
    [x, x, x], y,
    validation_data=([x_val, x_val, x_val], y_val),
    epochs = 3,
    batch_size = 3
)

Epoch 1/3
34/34 ━━━━━━━━━━━━━━━━━━━━ 479s 14s/step - accuracy: 0.5800 - loss: 0.8260 - val_accuracy: 0.3500 - val_loss: 0.8568
Epoch 2/3
34/34 ━━━━━━━━━━━━━━━━━━━━ 452s 13s/step - accuracy: 0.4300 - loss: 0.8553 - val_accuracy: 0.6500 - val_loss: 0.6347
Epoch 3/3
34/34 ━━━━━━━━━━━━━━━━━━━━ 446s 13s/step - accuracy: 0.6100 - loss: 0.6596 - val_accuracy: 0.6000 - val_loss: 0.6792


In [ ]:
loss, acc = target_model.evaluate([x_val, x_val, x_val], y_val, verbose=0)
print(f"Test accuracy: {acc * 100:2f}%")

Test accuracy: 60.000002%
